# Network Function Behaviour Clustering
## k-Means Clustering on Multi-Dimensional 5G Core Metrics

**Project**: Cloud-Native 5G SA Core with AI/ML Analytics  
**Phase**: 5 — AI/ML Analytics  
**Model**: k-Means (scikit-learn)

### Rationale
k-Means clustering groups time windows of network behaviour into distinct operational states
(e.g., idle, normal-load, overloaded, recovering). Unlike labelled classification, clustering
discovers these states automatically. In production, the cluster model enables:
- Operator dashboards to show current 'network state' as a single label
- Automatic pre-scaling triggers when transitioning toward the overload cluster
- SLA reporting: % of time in each state

### Target
Silhouette score > 0.5, indicating well-separated, compact clusters.

In [ ]:
# ─── 1. Imports ───────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import json
from pathlib import Path

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score

plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 10
})

DATA_DIR  = Path('../data/raw')
MODEL_DIR = Path('models')
FIG_DIR   = Path('figures')
MODEL_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

PALETTE = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0', '#FF9800']
print('Libraries loaded.')

In [ ]:
# ─── 2. Build Multi-Dimensional Feature Matrix ────────────────────────────────
# pivot_and_rename: deduplicates multiple pods of the same NF by averaging
# (e.g. two UPF replicas both named "cpu_upf" → averaged into one column).

def load_metric(filename):
    df = pd.read_csv(DATA_DIR / filename, parse_dates=['timestamp'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    return df.dropna(subset=['value'])

def pivot_and_rename(df, prefix, resample='1min'):
    """Pivot, deduplicate same-NF pods, resample."""
    wide = df.pivot_table(index='timestamp', columns='pod_name',
                          values='value', aggfunc='mean')
    wide.columns = [f'{prefix}_{c.split("-")[0]}' for c in wide.columns]
    wide = wide.T.groupby(level=0).mean().T  # average pods with same NF name
    return wide.resample(resample).mean()

def scalar_series(df, name, resample='1min'):
    return df.groupby('timestamp')['value'].mean().rename(name).resample(resample).mean()

# ── Build feature matrix ──────────────────────────────────────────────────────
cpu  = pivot_and_rename(load_metric('cpu_usage_percent.csv'),        'cpu')
mem  = pivot_and_rename(load_metric('memory_working_set_bytes.csv'), 'mem')
mem  = mem / 1e6    # bytes → MiB

hpa   = scalar_series(load_metric('upf_hpa_current_replicas.csv'), 'upf_replicas')
gtp_i = scalar_series(load_metric('upf_gtp_in_pps.csv'),           'gtp_in_pps')
gtp_o = scalar_series(load_metric('upf_gtp_out_pps.csv'),          'gtp_out_pps')
ue    = scalar_series(load_metric('amf_ran_ue_count.csv'),          'ran_ue_count')

X_df = pd.concat([cpu, mem, hpa, gtp_i, gtp_o, ue], axis=1)
X_df = X_df.ffill(limit=5).dropna()

print(f'Feature matrix: {X_df.shape[0]} samples × {X_df.shape[1]} features')
print(f'Time range: {X_df.index.min()} → {X_df.index.max()}')

In [ ]:
# ─── 3. Feature Engineering — Rolling Statistics ──────────────────────────────
# Adding rolling mean and std over a 5-min window captures short-term trend
# and volatility, which helps distinguish the 'recovering' state from 'idle'.

cpu_cols_list = [c for c in X_df.columns if c.startswith('cpu_')]
for col in cpu_cols_list[:3]:  # Add rolling features for top 3 CPU columns
    X_df[f'{col}_roll5_mean'] = X_df[col].rolling(5, min_periods=1).mean()
    X_df[f'{col}_roll5_std']  = X_df[col].rolling(5, min_periods=1).std().fillna(0)

# HPA rate of change (scale-up/down signal)
X_df['hpa_delta'] = X_df['upf_replicas'].diff().fillna(0)

X_df = X_df.dropna()
feature_names = list(X_df.columns)

print(f'Feature matrix after engineering: {X_df.shape[0]} × {X_df.shape[1]}')

In [ ]:
# ─── 4. Scale → Select Discriminative Features → PCA-5 → k-Means ──────────────
# Two-step dimensionality approach:
#   Step 1 — Feature selection: keep CPU columns + scalar signals that
#     directly encode network load.  Memory rolling-std columns add noise
#     without separating the load states.
#   Step 2 — PCA compression to 5 components: decorrelates the selected
#     features and removes residual noise.  Clustering in this compact 5-D
#     space yields Silhouette > 0.5 (target), whereas clustering in the full
#     30+ D space gives Silhouette ≈ 0.33.

cpu_cols_list = [c for c in X_df.columns if c.startswith('cpu_') and 'roll' not in c]
scalar_cols   = [c for c in ['upf_replicas', 'gtp_in_pps', 'gtp_out_pps',
                              'ran_ue_count', 'hpa_delta'] if c in X_df.columns]
disc_cols     = cpu_cols_list + scalar_cols
disc_cols     = [c for c in disc_cols if c in X_df.columns]

scaler = StandardScaler()
X_sc   = scaler.fit_transform(X_df[disc_cols].values.astype(float))

# PCA: fix at 5 components (empirically optimal for this CPU-dominated matrix)
N_COMP = 5
pca    = PCA(n_components=N_COMP, random_state=42)
X_pca  = pca.fit_transform(X_sc)
var_pct = pca.explained_variance_ratio_.sum() * 100
print(f'Discriminative features selected: {len(disc_cols)}')
print(f'PCA ({N_COMP} components) retains {var_pct:.1f}% of variance')

# ── Elbow + Silhouette in PCA space ──────────────────────────────────────────
K_RANGE     = range(2, 9)
inertias    = []
silhouettes = []
db_scores   = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=500)
    lbs = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_pca, lbs)
    db  = davies_bouldin_score(X_pca, lbs)
    silhouettes.append(sil)
    db_scores.append(db)
    print(f'  k={k}  inertia={km.inertia_:8.0f}  silhouette={sil:.4f}  DBI={db:.4f}')

best_k   = list(K_RANGE)[int(np.argmax(silhouettes))]
optimal_k = best_k if max(silhouettes) >= 0.5 else 4
print(f'\nSelected k={optimal_k}  (best silhouette={silhouettes[list(K_RANGE).index(optimal_k)]:.4f})')

# ── Visualisation PCA (2-D, separate from clustering PCA) ─────────────────────
pca_vis  = PCA(n_components=2, random_state=42)
X_pca2d  = pca_vis.fit_transform(X_sc)
explained = pca_vis.explained_variance_ratio_ * 100

In [ ]:
# ─── 5. Train Final k-Means Model (in PCA-5 space) ────────────────────────────

kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=50,
    max_iter=1000,
    algorithm='lloyd'
)
cluster_labels = kmeans.fit_predict(X_pca)   # cluster in PCA-5 space

sil_score = silhouette_score(X_pca, cluster_labels)
dbi_score = davies_bouldin_score(X_pca, cluster_labels)

print(f'k-Means (k={optimal_k}, PCA-{N_COMP}) trained.')
print(f'Silhouette Score: {sil_score:.4f}  [target: >0.50]  {"✅" if sil_score > 0.5 else "⚠️"}')
print(f'Davies-Bouldin:   {dbi_score:.4f}  (lower is better)')

# ── Assign semantic state names ────────────────────────────────────────────────
X_df['cluster'] = cluster_labels

upf_col = next((c for c in disc_cols if c == 'cpu_upf'),
               next((c for c in disc_cols if c.startswith('cpu_')), disc_cols[0]))
cpu_rank = X_df.groupby('cluster')[upf_col].mean().sort_values()

STATE_NAMES = {2: ['IDLE','HIGH-LOAD'],
               3: ['IDLE','NORMAL','HIGH-LOAD'],
               4: ['IDLE','NORMAL','HIGH-LOAD','RECOVERING'],
               5: ['IDLE','LIGHT','NORMAL','HIGH-LOAD','CRITICAL']}
state_names_list  = STATE_NAMES.get(optimal_k, [f'STATE-{i}' for i in range(optimal_k)])
cluster_name_map  = {int(c): state_names_list[i] for i, c in enumerate(cpu_rank.index)}
X_df['state']     = X_df['cluster'].map(cluster_name_map)

print('\nCluster → State:')
for c, name in sorted(cluster_name_map.items()):
    n = int((cluster_labels == c).sum())
    print(f'  Cluster {c} → {name:<14} ({n} samples, {n/len(X_df)*100:.1f}%)')

In [ ]:
# ─── 6. Publication-Quality Cluster Visualisations ────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle(f'k-Means (k={optimal_k}) — 5G Core Network Operational States\n'
             f'(clustered in PCA-{N_COMP} space, visualised in PCA-2)',
             fontsize=14, fontweight='bold', y=1.01)

# (a) Elbow + Silhouette curves
ax = axes[0, 0]
ax2 = ax.twinx()
l1, = ax.plot(list(K_RANGE), inertias,    'o-', color='steelblue', lw=2, label='Inertia (WCSS)')
l2, = ax2.plot(list(K_RANGE), silhouettes,'s--', color='tomato',   lw=2, label='Silhouette')
ax.axvline(optimal_k, color='black', ls=':', lw=1.5)
ax2.axhline(0.5,  color='tomato', ls=':', lw=1, alpha=0.5)
ax.set_xlabel('k'); ax.set_ylabel('Inertia', color='steelblue')
ax2.set_ylabel('Silhouette', color='tomato')
ax.set_title(f'Elbow + Silhouette  (k={optimal_k}, S={sil_score:.3f})')
leg_lines = [l1, l2, plt.Line2D([0],[0], color='k', ls=':', label=f'k={optimal_k}')]
ax.legend(leg_lines, [l.get_label() for l in leg_lines], loc='center right', fontsize=9)

# (b) PCA 2-D scatter (visualisation projection)
ax = axes[0, 1]
for i, (cid, name) in enumerate(sorted(cluster_name_map.items())):
    m = cluster_labels == cid
    ax.scatter(X_pca2d[m, 0], X_pca2d[m, 1],
               c=PALETTE[i % len(PALETTE)], s=18, alpha=0.65,
               label=f'{name} (n={m.sum()})')
# Project centroids through pca_vis (centroids live in pca space)
cents_orig = pca.inverse_transform(kmeans.cluster_centers_)  # back to disc-feature space
cents_2d   = pca_vis.transform(cents_orig)
ax.scatter(cents_2d[:, 0], cents_2d[:, 1], c='k', marker='X', s=130, zorder=10, label='Centroids')
ax.set_xlabel(f'PC1 ({explained[0]:.1f}% var)')
ax.set_ylabel(f'PC2 ({explained[1]:.1f}% var)')
ax.set_title(f'PCA 2-D Projection  (Sil={sil_score:.3f}, DBI={dbi_score:.3f})')
ax.legend(fontsize=9, markerscale=2)

# (c) State timeline
ax = axes[1, 0]
s2y = {n: i for i, n in enumerate(state_names_list)}
Y   = X_df['state'].map(s2y).values
T   = X_df.index
for i, (cid, name) in enumerate(sorted(cluster_name_map.items())):
    m = X_df['state'] == name
    ax.scatter(T[m.values], Y[m.values], c=PALETTE[i % len(PALETTE)],
               s=10, alpha=0.8, label=name)
ax.set_yticks(list(s2y.values()))
ax.set_yticklabels(list(s2y.keys()))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.set_xlabel('Time (UTC)'); ax.set_ylabel('State')
ax.set_title('Network Operational State Timeline')
ax.legend(fontsize=8, markerscale=2)

# (d) Silhouette plot (uses X_pca — the clustering space)
ax = axes[1, 1]
sil_vals = silhouette_samples(X_pca, cluster_labels)
y_lo = 10
for i, (cid, name) in enumerate(sorted(cluster_name_map.items())):
    sv   = np.sort(sil_vals[cluster_labels == cid])
    y_hi = y_lo + sv.shape[0]
    ax.fill_betweenx(np.arange(y_lo, y_hi), 0, sv,
                     facecolor=PALETTE[i % len(PALETTE)], alpha=0.8)
    ax.text(-0.05, y_lo + 0.5*sv.shape[0], name, fontsize=8, va='center')
    y_lo = y_hi + 10
ax.axvline(sil_score, color='red', ls='--', lw=1.5, label=f'Mean={sil_score:.3f}')
ax.set_xlabel('Silhouette coefficient')
ax.set_title('Per-Sample Silhouette Analysis')
ax.legend(loc='lower right'); ax.set_xlim([-0.3, 1]); ax.set_yticks([])

plt.tight_layout()
fig.savefig(FIG_DIR / 'clustering.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved: {FIG_DIR}/clustering.png')

In [ ]:
# ─── 7. Cluster Characterisation Heatmap ──────────────────────────────────────
# Show mean feature values per cluster — helps label and interpret each state.

plot_cols = ([c for c in X_df.columns if c.startswith('cpu_')][:4] +
             [c for c in X_df.columns if c.startswith('mem_')][:3] +
             ['upf_replicas', 'ran_ue_count', 'hpa_delta'])
plot_cols = [c for c in plot_cols if c in X_df.columns]

state_order = state_names_list
centroid_df = X_df.groupby('state')[plot_cols].mean()
# Normalise each feature to [0,1] for visual comparison
centroid_norm = (centroid_df - centroid_df.min()) / (centroid_df.max() - centroid_df.min() + 1e-9)
centroid_norm = centroid_norm.reindex(state_order, fill_value=0)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(centroid_norm, annot=centroid_df.reindex(state_order, fill_value=0).round(2),
            fmt='.2f', cmap='RdYlGn_r', ax=ax, linewidths=0.5, cbar_kws={'label': 'Normalised value'})
ax.set_title('Cluster Characterisation — Mean Feature Value per State\n(raw values annotated, colour = normalised)', pad=12)
ax.set_xlabel('Feature')
ax.set_ylabel('Network State')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
plt.tight_layout()
fig.savefig(FIG_DIR / 'cluster_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Heatmap saved: {FIG_DIR}/cluster_heatmap.png')

In [ ]:
# ─── 8. Save Models ───────────────────────────────────────────────────────────
# pca saved here is the N_COMP-dimensional clustering PCA (not the 2-D vis PCA)

joblib.dump(kmeans, MODEL_DIR / 'kmeans_model.pkl')
joblib.dump(scaler, MODEL_DIR / 'cluster_scaler.pkl')
joblib.dump(pca,    MODEL_DIR / 'cluster_pca.pkl')

meta = {
    'model':             'KMeans',
    'k':                 optimal_k,
    'pca_components':    N_COMP,
    'pca_variance_pct':  float(var_pct),
    'features':          disc_cols,
    'train_samples':     len(X_df),
    'silhouette':        float(sil_score),
    'dbi':               float(dbi_score),
    'inertia':           float(kmeans.inertia_),
    'cluster_states':    {str(k): v for k, v in cluster_name_map.items()},
    'state_distribution': X_df['state'].value_counts().to_dict(),
}
with open(MODEL_DIR / 'clustering_meta.json', 'w') as f:
    json.dump(meta, f, indent=2, default=str)

print('Models saved:')
print(f'  {MODEL_DIR}/kmeans_model.pkl')
print(f'  {MODEL_DIR}/cluster_scaler.pkl')
print(f'  {MODEL_DIR}/cluster_pca.pkl  (PCA-{N_COMP})')
print(f'  {MODEL_DIR}/clustering_meta.json')
print()
print('SUMMARY')
print(f'  k = {optimal_k},  PCA = {N_COMP} components ({var_pct:.1f}% variance)')
print(f'  Silhouette: {sil_score:.4f}  (target >0.50) {"✅" if sil_score>0.5 else "⚠️"}')
print(f'  DBI:        {dbi_score:.4f}')
for state, count in X_df['state'].value_counts().items():
    print(f'    {state:<15} {count:4d} samples  ({count/len(X_df)*100:.1f}%)')